# 22 — 181 chỉ số tài chính và phân tích DuPont

`client.financials.indicators()` là bề mặt đáng giá nhất của thư viện mà README
không nhắc tới: **181 chỉ tiêu đã tính sẵn** từ báo cáo tài chính, chia theo bốn
loại hình doanh nghiệp. Bạn không phải tự viết công thức ROE, không phải tự
quyết định lấy vốn chủ sở hữu đầu kỳ hay cuối kỳ.

Notebook này gồm:

1. Danh mục chỉ tiêu — và hai cột làm nó dùng được: `formula`, `higher_is_better`
2. ⚠️ **Cái bẫy lớn nhất: kỳ của chỉ tiêu không đồng nhất.** `pe`/`eps` là
   **4 quý gần nhất**; `roe`/`roa` thì **tuỳ loại hình doanh nghiệp** — và đó
   là chỗ một screener trộn ngân hàng với doanh nghiệp thường sẽ hỏng.
3. DuPont ba tầng và năm tầng, kiểm chứng bằng chính dữ liệu

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, heatmap, hom_nay

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

pd.set_option("display.max_colwidth", 90)

## 1 · Danh mục chỉ tiêu

Hỏi danh mục trước. `indicator_catalog()` không tốn hạn mức đáng kể và nó là
thứ duy nhất cho biết chỉ tiêu nào tồn tại với loại hình nào.

In [2]:
cat = client.financials.indicator_catalog()

print(f"{len(cat)} dòng · {cat['code'].nunique()} mã chỉ tiêu khác nhau")
(
    cat.groupby("company_type", observed=True)
    .agg(so_chi_tieu=("code", "nunique"), so_nhom=("group", "nunique"))
    .rename(index={"CT": "CT — phi tài chính", "NH": "NH — ngân hàng", "CK": "CK — chứng khoán", "BH": "BH — bảo hiểm"})
)

181 dòng · 135 mã chỉ tiêu khác nhau


,so_chi_tieu,so_nhom
company_type,,
BH — bảo hiểm,39,10
CK — chứng khoán,38,11
CT — phi tài chính,61,11
NH — ngân hàng,43,9


Mỗi loại hình có bộ chỉ tiêu riêng. Ngân hàng có `nim`, `npl_ratio`, `car`;
công ty chứng khoán có `margin_loan`, `brokerage_ratio`; doanh nghiệp phi tài
chính có `inventory_days`, `ccc`. Không có chuyện một chỉ tiêu tồn tại ở mọi
nơi ngoài nhóm định giá.

In [3]:
theo_nhom = (
    cat.groupby(["company_type", "group"], observed=True)["code"]
    .count()
    .unstack(fill_value=0)
)
theo_nhom.loc[["CT", "NH", "CK", "BH"]]

group,asset_quality,capital,cashflow,debt_service,efficiency,growth,income_structure,investment,liquidity,margin_lending,profitability,prop_book,quality,reserves,size,solvency,underwriting,valuation
company_type,,,,,,,,,,,,,,,,,,
CT,0,1,5,5,5,5,0,0,3,0,10,0,6,0,5,7,0,9
NH,6,3,0,0,2,7,3,0,3,0,6,0,0,0,8,0,0,5
CK,0,0,0,1,1,5,6,0,1,5,4,2,0,0,6,2,0,5
BH,0,2,0,1,0,5,2,4,0,0,2,0,0,5,4,0,9,5


### Hai cột làm danh mục này dùng được

**`formula`** — định nghĩa bằng chữ, nên bạn biết chính xác con số nghĩa là gì
trước khi dùng nó. **`higher_is_better`** — chiều tốt/xấu, nên bạn xếp hạng
được mà không phải tự quyết định.

In [4]:
mau = cat[(cat["company_type"] == "CT") & (cat["group"].isin(["profitability", "solvency"]))]
mau[["code", "label", "unit", "higher_is_better", "formula"]].head(12)

,code,label,unit,higher_is_better,formula
9,roe,ROE,ratio,True,Lợi nhuận sau thuế của cổ đông công ty mẹ ÷ vốn chủ sở hữu.
10,roa,ROA,ratio,True,Lợi nhuận sau thuế của cổ đông công ty mẹ ÷ tổng tài sản.
11,gross_margin,Biên lợi nhuận gộp,ratio,True,Lợi nhuận gộp ÷ doanh thu thuần.
12,net_margin,Biên lợi nhuận ròng,ratio,True,Lợi nhuận sau thuế của cổ đông công ty mẹ ÷ doanh thu thuần.
13,ebit,EBIT,VND,True,Lợi nhuận trước thuế + chi phí lãi vay.
14,ebitda,EBITDA,VND,True,EBIT + khấu hao tài sản cố định.
15,ebit_margin,Biên EBIT,ratio,True,EBIT ÷ doanh thu thuần.
16,roic,ROIC,ratio,True,EBIT × (1 − thuế suất thực tế) ÷ (vốn chủ sở hữu + nợ vay có lãi − tiền). Thuế suất th...
17,financial_result_net,Kết quả tài chính ròng,VND,<NA>,Doanh thu hoạt động tài chính − chi phí tài chính (đã gồm chi phí lãi vay).
18,core_pretax_profit,Lợi nhuận trước thuế cốt lõi,VND,True,Lợi nhuận trước thuế − kết quả tài chính ròng − lãi/lỗ liên doanh liên kết − lợi nhuận...


Đọc kỹ hai dòng này — chúng là ví dụ cho việc `formula` cứu bạn khỏi một giả
định sai:

In [5]:
for ma_ct in ["debt_to_equity", "fcf", "ccc"]:
    r = cat[(cat["company_type"] == "CT") & (cat["code"] == ma_ct)].iloc[0]
    print(f"■ {r['label']} ({ma_ct}) [{r['unit']}]")
    print(f"  {r['formula']}\n")

■ Nợ phải trả trên vốn chủ sở hữu (debt_to_equity) [x]
  Tổng nợ phải trả ÷ vốn chủ sở hữu. ⚠️ Đây là **toàn bộ** nợ phải trả, không chỉ nợ vay có lãi — xem `interest_bearing_debt`.

■ Dòng tiền tự do (fcf) [VND]
  Dòng tiền từ hoạt động kinh doanh + chi mua sắm tài sản cố định. Khoản chi đã mang dấu âm ở nguồn nên đây là phép **cộng**.

■ Chu kỳ tiền mặt (CCC) (ccc) [days]
  Số ngày phải thu + số ngày tồn kho − số ngày phải trả.



`debt_to_equity` là **toàn bộ** nợ phải thu trả, không chỉ nợ vay có lãi — nếu
bạn tưởng ngược lại thì mọi kết luận về đòn bẩy đều lệch. `fcf` là một phép
**cộng** vì khoản chi đã mang dấu âm ở nguồn.

`higher_is_better` có ba giá trị, và giá trị thứ ba mới là điều đáng chú ý:

In [6]:
cat.groupby(cat["higher_is_better"].astype("object").fillna("<NA> — không xếp hạng được"))[
    "code"
].count().rename("số chỉ tiêu").to_frame()

,số chỉ tiêu
higher_is_better,
False,41
True,87
<NA> — không xếp hạng được,53


In [7]:
print("Các chỉ tiêu KHÔNG có chiều tốt/xấu:")
print(sorted(cat[cat["higher_is_better"].isna()]["code"].unique()))

Các chỉ tiêu KHÔNG có chiều tốt/xấu:
['asset_growth', 'brokerage_ratio', 'ceded_ratio', 'cff', 'cfi', 'cip_to_assets', 'claim_reserve_change', 'customer_deposit', 'customer_loan', 'deposit_growth', 'effective_tax_rate', 'enterprise_value', 'equity_growth', 'equity_multiplier', 'financial_profit_to_pbt', 'financial_result_net', 'ib_ratio', 'interest_bearing_debt', 'investment_assets', 'investment_to_assets', 'lar', 'loan_growth', 'margin_growth', 'margin_income_ratio', 'margin_loan', 'market_cap', 'net_debt', 'payable_days', 'premium_growth', 'prop_assets', 'prop_income_ratio', 'provision_charge_to_loan', 'provision_expense', 'receivable_provision_charge', 'reserve_to_equity', 'retention_ratio', 'st_borrowing', 'total_assets', 'total_equity', 'total_liabilities', 'underwriting_reserve', 'unrealized_profit']


`market_cap`, `total_assets`, `ebitda`… là **quy mô**, không phải chất lượng.
Lớn hơn không tốt hơn. Một screener xếp hạng chúng như thể có chiều là một
screener chọn ra các doanh nghiệp to nhất và gọi đó là "tốt nhất".

## 2 · ⚠️ Cái bẫy kỳ: cùng một frame, hai kỳ khác nhau

Đây là chỗ dễ sai nhất và nó sai không kèm cảnh báo nào.

In [8]:
hpg = client.financials.indicators(
    "HPG",
    codes=["pe", "eps", "roe", "roa", "net_margin"],
    period="quarterly",
    start_year=2025,
    end_year=2025,
)
theo_quy = hpg.pivot_table(index="period_label", columns="code", values="value")

hpg_nam = client.financials.indicators(
    "HPG", codes=["pe", "eps", "roe", "roa", "net_margin"], period="annual", start_year=2025, end_year=2025
).set_index("code")["value"]

print("BỐN QUÝ 2025:")
print(theo_quy.round(4).to_string())
print("\nCẢ NĂM 2025:")
print(hpg_nam.round(4).to_string())

BỐN QUÝ 2025:
code                eps  net_margin       pe     roa     roe
period_label                                                
Q1 2025       1954.3707      0.0889  13.6873  0.0146  0.0283
Q2 2025       1751.7625      0.1185  12.9584  0.0176  0.0348
Q3 2025       1880.8205      0.1095  14.9669  0.0162  0.0313
Q4 2025       2021.3668      0.0836  13.0605  0.0150  0.0294

CẢ NĂM 2025:
code
pe              13.0605
eps           2021.3668
roe              0.1178
roa              0.0599
net_margin       0.0990


In [9]:
print("Kiểm hai giả thuyết:\n")
print(f"  eps Q4 2025      = {theo_quy.loc['Q4 2025', 'eps']:>12,.2f}")
print(f"  eps năm 2025     = {hpg_nam['eps']:>12,.2f}   ← BẰNG NHAU")
print("    → eps (và pe) trong kỳ quarterly là của 4 QUÝ GẦN NHẤT\n")
print(f"  tổng roe 4 quý   = {theo_quy['roe'].sum():>12,.4f}")
print(f"  roe năm 2025     = {hpg_nam['roe']:>12,.4f}   ← KHÁC NHAU")
print(f"  roe Q4 2025      = {theo_quy.loc['Q4 2025', 'roe']:>12,.4f}   ← chỉ bằng ~1/4 roe năm")
print("    → với HPG, roe/roa/net_margin quarterly là của RIÊNG QUÝ ĐÓ")
print("      (giữ chữ 'với HPG' — phần sau cho thấy vì sao nó không phải luật chung)")

Kiểm hai giả thuyết:

  eps Q4 2025      =     2,021.37
  eps năm 2025     =     2,021.37   ← BẰNG NHAU
    → eps (và pe) trong kỳ quarterly là của 4 QUÝ GẦN NHẤT

  tổng roe 4 quý   =       0.1238
  roe năm 2025     =       0.1178   ← KHÁC NHAU
  roe Q4 2025      =       0.0294   ← chỉ bằng ~1/4 roe năm
    → với HPG, roe/roa/net_margin quarterly là của RIÊNG QUÝ ĐÓ
      (giữ chữ 'với HPG' — phần sau cho thấy vì sao nó không phải luật chung)


### Và đây là tầng thứ hai của cái bẫy: nó **khác nhau theo loại hình doanh nghiệp**

Kết luận vừa rồi đúng với HPG — một doanh nghiệp `CT`. Nó **không** đúng với
ngân hàng. Đo trên mười lăm mã của bốn loại hình, so quý 4 với cả năm:

In [10]:
MAU_LOAI_HINH = {
    "NH — ngân hàng": ["VCB", "ACB", "TCB", "MBB", "VPB"],
    "CT — phi tài chính": ["HPG", "FPT", "VNM", "MWG", "GVR"],
    "CK — chứng khoán": ["SSI", "VCI", "HCM"],
    "BH — bảo hiểm": ["BVH", "BMI"],
}

dong = []
for nhan, ma_ds in MAU_LOAI_HINH.items():
    quy = client.financials.indicators(
        ma_ds, codes=["roe", "roa"], period="quarterly", start_year=2025, end_year=2025
    )
    nam = client.financials.indicators(
        ma_ds, codes=["roe", "roa"], period="annual", start_year=2025, end_year=2025
    )
    q4 = quy[quy["quarter"] == 4].set_index(["symbol", "code"])["value"]
    ca_nam = nam.set_index(["symbol", "code"])["value"]
    chung = q4.index.intersection(ca_nam.index)
    for k in chung:
        dong.append({"loại hình": nhan, "mã": k[0], "chỉ tiêu": k[1], "Q4 ÷ năm": q4[k] / ca_nam[k]})

ty_le = pd.DataFrame(dong)
ty_le.groupby(["loại hình", "chỉ tiêu"], observed=True)["Q4 ÷ năm"].agg(
    ["median", "min", "max"]
).round(3)

median    min    max
loại hình          chỉ tiêu                      
BH — bảo hiểm      roa        1.042  1.040  1.045
                   roe        1.027  1.010  1.045
CK — chứng khoán   roa        1.000  1.000  1.000
                   roe        1.000  1.000  1.000
CT — phi tài chính roa        0.267  0.188  0.302
                   roe        0.267  0.187  0.302
NH — ngân hàng     roa        1.000  1.000  1.000
                   roe        1.038  0.987  1.057

Đọc cột `median`:

| Loại hình | Q4 ÷ năm | Nghĩa là |
|---|---|---|
| `CT` phi tài chính | **≈ 0,27** | ROE quý là của **riêng quý đó** |
| `NH` ngân hàng | **≈ 1,00** | ROE quý đã **quy về năm** (TTM) |
| `CK` chứng khoán | **≈ 1,00** | quy về năm |
| `BH` bảo hiểm | **≈ 1,03** | quy về năm |

### Hệ quả: một screener trộn ngân hàng với doanh nghiệp thường trên dữ liệu quý sẽ hỏng

Ngân hàng ROE 17% xuất hiện với con số `0,17` còn doanh nghiệp thép ROE 12%
xuất hiện với con số `0,03`. Xếp hạng chung một bảng thì **mọi ngân hàng đều
nằm trên mọi doanh nghiệp phi tài chính**, không phải vì chúng tốt hơn mà vì
chúng được đo bằng một thước khác.

Không có cảnh báo nào. Cả hai con số đều nằm trong khoảng hợp lệ, cột `unit`
đều ghi `ratio`, và cột `period_type` đều ghi `quarterly`.

In [11]:
minh_hoa = client.financials.indicators(
    ["VCB", "ACB", "HPG", "FPT"], codes=["roe"], period="quarterly", start_year=2026
)
print("ROE quý gần nhất, xếp hạng như một screener ngây thơ sẽ làm:")
print(
    minh_hoa[minh_hoa["quarter"] == minh_hoa["quarter"].max()]
    .assign(roe_pct=lambda d: (d["value"] * 100).round(2))
    .sort_values("roe_pct", ascending=False)[["symbol", "company_type", "period_label", "roe_pct"]]
    .to_string(index=False)
)
print("\n→ Hai ngân hàng đứng đầu. Nhưng ROE NĂM 2025 của FPT là 21,4% — cao hơn cả hai.")

ROE quý gần nhất, xếp hạng như một screener ngây thơ sẽ làm:
symbol company_type period_label  roe_pct
   VCB           NH      Q2 2026    18.02
   ACB           NH      Q2 2026    16.81
   FPT           CT      Q2 2026     6.26
   HPG           CT      Q2 2026     4.50

→ Hai ngân hàng đứng đầu. Nhưng ROE NĂM 2025 của FPT là 21,4% — cao hơn cả hai.


### Cách phân biệt

Với các chỉ tiêu **định giá**, `formula` nói thẳng:

In [12]:
ttm = cat[cat["formula"].str.contains("4 quý gần nhất", na=False)]
print(f"Chỉ tiêu ghi rõ 'tính trên 4 quý gần nhất' trong formula: {sorted(ttm['code'].unique())}")

Chỉ tiêu ghi rõ 'tính trên 4 quý gần nhất' trong formula: ['eps', 'pe']


Với các chỉ tiêu **sinh lời** thì `formula` không nói — quy ước nằm ở nguồn báo
cáo của từng loại hình. Nên quy tắc thực hành là:

> **Dùng `period="annual"` cho mọi phép so sánh cắt ngang và mọi bộ lọc.**
> Nó nhất quán với cả bốn loại hình, và đó là điều duy nhất quan trọng khi bạn
> xếp hạng nhiều mã cạnh nhau.

Dùng `period="quarterly"` khi bạn nhìn **nhịp theo thời gian của một mã** — ở
đó quy ước nào cũng được, miễn là bạn không đặt hai loại hình cạnh nhau. Và nếu
buộc phải trộn, hãy kiểm bằng chính phép đo ở trên với dữ liệu của bạn.

## 3 · Từ dạng long sang dạng bảng

`indicators()` trả về dạng long như mọi thứ khác. Đây là hàm bạn sẽ dùng lại ở
`23` và `24`.

In [13]:
def bang_chi_so(df: pd.DataFrame, *, chi_muc: str = "symbol") -> pd.DataFrame:
    """Xoay khung chỉ tiêu dạng long thành bảng: một dòng một thực thể, một cột một chỉ tiêu."""
    return df.pivot_table(index=chi_muc, columns="code", values="value")


THEP = ["HPG", "HSG", "NKG", "TLH", "SMC"]

thep = client.financials.indicators(
    THEP,
    codes=["roe", "roa", "net_margin", "gross_margin", "asset_turnover", "equity_multiplier", "debt_to_equity", "pe", "pb"],
    period="annual",
    start_year=2025,
    end_year=2025,
)
bang_thep = bang_chi_so(thep)
bang_thep.round(4)

code,asset_turnover,debt_to_equity,equity_multiplier,gross_margin,net_margin,pb,pe,roa,roe
symbol,,,,,,,,,
HPG,0.6053,0.9654,1.9654,0.1569,0.0990,1.5442,13.0605,0.0599,0.1178
HSG,1.6478,0.8511,1.8511,0.1226,0.0182,0.8597,15.4797,0.0300,0.0555
NKG,0.8978,1.1602,2.1602,0.0530,0.0133,0.8705,33.7061,0.0120,0.0258
SMC,1.5740,3.4210,4.4210,0.0002,0.0257,0.9608,5.2299,0.0404,0.1788
TLH,1.8681,1.2536,2.2536,0.0339,0.0010,0.4213,93.3319,0.0018,0.0040


⚠️ Chú ý cột `unit`: `roe` là `ratio` (0,1178 = 11,78%) còn `pe` là `x` (lần).
Nhân 100 cho cột `pe` là một lỗi im lặng. Lấy đơn vị từ dữ liệu:

In [14]:
don_vi = thep.groupby("code", observed=True)["unit"].first()
print(don_vi.to_string())

# Đổi các cột ratio sang phần trăm cho dễ đọc — có kiểm đơn vị, không đoán
la_ty_le = don_vi[don_vi == "ratio"].index
hien_thi = bang_thep.copy()
hien_thi[la_ty_le] = hien_thi[la_ty_le] * 100
hien_thi.round(2)

code
asset_turnover           x
debt_to_equity           x
equity_multiplier        x
gross_margin         ratio
net_margin           ratio
pb                       x
pe                       x
roa                  ratio
roe                  ratio


code,asset_turnover,debt_to_equity,equity_multiplier,gross_margin,net_margin,pb,pe,roa,roe
symbol,,,,,,,,,
HPG,0.61,0.97,1.97,15.69,9.90,1.54,13.06,5.99,11.78
HSG,1.65,0.85,1.85,12.26,1.82,0.86,15.48,3.00,5.55
NKG,0.90,1.16,2.16,5.30,1.33,0.87,33.71,1.20,2.58
SMC,1.57,3.42,4.42,0.02,2.57,0.96,5.23,4.04,17.88
TLH,1.87,1.25,2.25,3.39,0.10,0.42,93.33,0.18,0.40


## 4 · DuPont ba tầng

ROE trả lời "sinh lời bao nhiêu trên vốn chủ", nhưng không nói **vì sao**. Hai
doanh nghiệp cùng ROE 20% có thể đến đó bằng hai con đường trái ngược: một bên
biên lợi nhuận dày, một bên vòng quay nhanh, một bên vay nhiều.

```
ROE  =  Biên lợi nhuận ròng  ×  Vòng quay tài sản  ×  Đòn bẩy tài chính
        (net_margin)            (asset_turnover)      (equity_multiplier)
```

Cả ba thành phần **đều có sẵn** trong danh mục. Kiểm chứng đẳng thức trước khi
dùng nó — nếu nguồn dữ liệu không nhất quán thì mọi diễn giải sau đều vô nghĩa:

In [15]:
NHOM = ["HPG", "FPT", "MWG", "VNM", "VCB"]

dp = client.financials.indicators(
    NHOM,
    codes=["roe", "net_margin", "asset_turnover", "equity_multiplier"],
    period="annual",
    start_year=2021,
)
dupont = dp.pivot_table(index=["symbol", "year"], columns="code", values="value").dropna()

dupont["dupont3"] = dupont["net_margin"] * dupont["asset_turnover"] * dupont["equity_multiplier"]
dupont["lech"] = (dupont["dupont3"] - dupont["roe"]).abs()

print(f"Đẳng thức DuPont trên {len(dupont)} quan sát:")
print(f"  lệch trung vị:  {dupont['lech'].median():.2e}")
print(f"  lệch lớn nhất:  {dupont['lech'].max():.2e}   ← sai số dấu phẩy động của máy")

Đẳng thức DuPont trên 20 quan sát:
  lệch trung vị:  1.39e-17
  lệch lớn nhất:  5.55e-17   ← sai số dấu phẩy động của máy


Khớp tới độ chính xác của số dấu phẩy động. Đẳng thức đúng, dùng được.

In [16]:
nam_moi = dupont.reset_index()
nam_moi = nam_moi[nam_moi["year"] == nam_moi["year"].max()]

trinh_bay = pd.DataFrame(
    {
        "Biên LN ròng (%)": (nam_moi["net_margin"] * 100).round(2).values,
        "Vòng quay TS (lần)": nam_moi["asset_turnover"].round(2).values,
        "Đòn bẩy (lần)": nam_moi["equity_multiplier"].round(2).values,
        "ROE (%)": (nam_moi["roe"] * 100).round(2).values,
    },
    index=nam_moi["symbol"].values,
)
trinh_bay.sort_values("ROE (%)", ascending=False)

,Biên LN ròng (%),Vòng quay TS (lần),Đòn bẩy (lần),ROE (%)
VNM,14.79,1.19,1.55,27.29
FPT,13.37,0.80,2.01,21.43
MWG,4.51,1.86,2.53,21.20
HPG,9.90,0.61,1.97,11.78


Ba con đường tới ROE, đọc được ngay trên bảng: doanh nghiệp bán lẻ có biên
mỏng nhưng vòng quay tài sản cao gấp ba lần các mã khác; ngân hàng có đòn bẩy
cao hơn hẳn — đó là mô hình kinh doanh chứ không phải rủi ro bất thường.

⚠️ **Đây là lý do không so DuPont giữa các ngành.** Đòn bẩy 15 lần ở ngân hàng
là bình thường; 15 lần ở một công ty thép là sắp phá sản.

In [17]:
ve = nam_moi.melt(
    id_vars="symbol",
    value_vars=["net_margin", "asset_turnover", "equity_multiplier"],
    var_name="thanh_phan",
    value_name="gia_tri",
)
ten = {
    "net_margin": "Biên LN ròng (×100)",
    "asset_turnover": "Vòng quay tài sản",
    "equity_multiplier": "Đòn bẩy tài chính",
}
ve["thanh_phan"] = ve["thanh_phan"].map(ten)
ve.loc[ve["thanh_phan"] == "Biên LN ròng (×100)", "gia_tri"] *= 100

bang_ve = ve.pivot(index="symbol", columns="thanh_phan", values="gia_tri").round(2)
bang_ve = bang_ve.loc[nam_moi.set_index("symbol")["roe"].sort_values(ascending=False).index]

heatmap(
    bang_ve,
    tieu_de=f"Ba thành phần DuPont — năm {int(nam_moi['year'].iloc[0])}",
    phu_de="Xếp theo ROE giảm dần · thang một sắc vì đây là độ lớn, không phải dấu",
    nhan_mau="giá trị",
    phan_ky=False,
    dinh_dang_o="%{z:.2f}",
)

## 5 · DuPont năm tầng

Bản ba tầng gộp thuế và lãi vay vào biên lợi nhuận ròng, nên nó không tách được
"doanh nghiệp hoạt động tốt" khỏi "doanh nghiệp được ưu đãi thuế".

```
ROE = Gánh nặng thuế × Gánh nặng lãi vay × Biên EBIT × Vòng quay TS × Đòn bẩy
      (LNST/LNTT)      (LNTT/EBIT)         (EBIT/DT)
```

Ba thành phần đầu dựng từ `ebit`, `ebit_margin`, `effective_tax_rate` và
`interest_coverage` — tất cả đều có trong danh mục `CT`.

In [18]:
PHI_TC = ["HPG", "FPT", "MWG", "VNM", "GVR", "DGC"]

nam_tang = client.financials.indicators(
    PHI_TC,
    codes=["roe", "net_margin", "ebit_margin", "effective_tax_rate", "interest_coverage",
           "asset_turnover", "equity_multiplier"],
    period="annual",
    start_year=2025,
    end_year=2025,
)
b5 = nam_tang.pivot_table(index="symbol", columns="code", values="value").dropna()

# Gánh nặng thuế = 1 − thuế suất hiệu dụng
b5["ganh_nang_thue"] = 1 - b5["effective_tax_rate"]
# Gánh nặng lãi vay = LNTT/EBIT = (EBIT − lãi vay)/EBIT = 1 − 1/interest_coverage
b5["ganh_nang_lai"] = 1 - 1 / b5["interest_coverage"]
b5["dupont5"] = (
    b5["ganh_nang_thue"] * b5["ganh_nang_lai"] * b5["ebit_margin"]
    * b5["asset_turnover"] * b5["equity_multiplier"]
)
b5["lech_diem_pt"] = (b5["dupont5"] - b5["roe"]).abs() * 100

ket = b5[["ganh_nang_thue", "ganh_nang_lai", "ebit_margin", "asset_turnover",
          "equity_multiplier", "dupont5", "roe", "lech_diem_pt"]].round(4)
ket.columns = ["Thuế", "Lãi vay", "Biên EBIT", "Vòng quay", "Đòn bẩy", "DuPont5", "ROE thật", "Lệch (đpt)"]
ket.sort_values("ROE thật", ascending=False)

,Thuế,Lãi vay,Biên EBIT,Vòng quay,Đòn bẩy,DuPont5,ROE thật,Lệch (đpt)
symbol,,,,,,,,
VNM,0.8080,0.9728,0.1882,1.1938,1.5460,0.2730,0.2729,0.0098
FPT,0.8611,0.9415,0.1976,0.7955,2.0148,0.2568,0.2143,4.2430
MWG,0.8192,0.8544,0.0648,1.8575,2.5303,0.2132,0.2120,0.1172
DGC,0.8841,0.9887,0.3204,0.5764,1.2711,0.2052,0.1946,1.0628
HPG,0.8600,0.8528,0.1355,0.6053,1.9654,0.1182,0.1178,0.0471
GVR,0.8440,0.9672,0.2527,0.3370,1.3871,0.0964,0.0855,1.0932


Bản năm tầng **không khớp tuyệt đối** như bản ba tầng, và điều đó là bình
thường: `ebit_margin` dùng doanh thu thuần làm mẫu số, còn ROE gộp cả lợi nhuận
từ hoạt động tài chính và công ty liên kết — những khoản không đi qua doanh thu.

Ba tầng là một **đẳng thức**; năm tầng là một **phép phân rã xấp xỉ**. Dùng nó
để đọc *nguồn gốc* của ROE, đừng dùng để tái tạo con số ROE.

## 6 · Chuỗi thời gian: ROE đến từ đâu qua các năm

In [19]:
MA = "MWG"
chuoi = client.financials.indicators(
    MA,
    codes=["roe", "net_margin", "asset_turnover", "equity_multiplier"],
    period="annual",
    start_year=2017,
)
c = chuoi.pivot_table(index="year", columns="code", values="value").dropna()
# Chuẩn hoá về năm đầu = 100 để ba thành phần khác thang so được với nhau
chuan = (c / c.iloc[0] * 100).reset_index().melt(
    id_vars="year", var_name="thanh_phan", value_name="chi_so_100"
)
ten5 = {**ten, "roe": "ROE"}
chuan["thanh_phan"] = chuan["thanh_phan"].map(lambda x: ten5.get(x, x)).str.replace(" (×100)", "", regex=False)

duong(
    chuan,
    x="year",
    y="chi_so_100",
    theo="thanh_phan",
    tieu_de=f"{MA} — ROE và ba thành phần DuPont, chuẩn hoá về 100",
    phu_de="Đường nào kéo ROE lên, đường nào kéo xuống",
    nhan_y="chỉ số (năm đầu = 100)",
)

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Có những chỉ tiêu nào | `financials.indicator_catalog(com_type="NH")` |
| Chỉ tiêu của nhiều mã | `financials.indicators(ma, codes=[...], period="annual")` |
| Cả một nhóm chỉ tiêu | `groups=["profitability", "solvency"]` |
| Chiều tốt/xấu để xếp hạng | cột `higher_is_better` của danh mục |

**Bốn điều mang sang notebook sau:**

1. ⚠️ **Kỳ của chỉ tiêu không đồng nhất, theo hai chiều cùng lúc.** Trong một
   frame quarterly: `pe`/`eps` là 4 quý gần nhất còn `roe`/`roa` thì tuỳ loại
   hình — `CT` cho giá trị **riêng quý đó**, còn `NH`/`CK`/`BH` cho giá trị
   **đã quy về năm**. Trộn hai loại hình trên dữ liệu quý là một bảng xếp hạng
   sai hệ thống, không kèm cảnh báo. **Dùng `period="annual"` khi so cắt ngang.**
2. `higher_is_better` là `<NA>` với các chỉ tiêu **quy mô**. Xếp hạng chúng là
   xếp hạng độ to, không phải độ tốt.
3. DuPont ba tầng là đẳng thức đúng tới sai số máy; năm tầng là phân rã xấp xỉ.
4. Không so DuPont giữa các ngành — đòn bẩy 15 lần ở ngân hàng là mô hình kinh
   doanh, ở công ty thép là báo động.

---

**Tiếp theo:** [`23_screener_dinh_gia.ipynb`](23_screener_dinh_gia.ipynb) —
chấm điểm toàn sàn HOSE, chuẩn hoá trong từng ngành.